# <center> **<span style="font-size:80px;">Físicos</span>** </center>
# MODELO PREDICTIVO HÍBRIDO POR BARRIOS
### FÍSICA (FOURIER) + COMPORTAMIENTO (MACHINE LEARNING)
---
Este notebook genera un modelo de demanda hídrica hiper-segmentado. Calcula una onda estacional independiente para cada barrio de Alicante y ajusta el comportamiento usando Machine Learning con datos reales de clima, satélite y viviendas turísticas.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

from sklearn.metrics import r2_score

# Tu configuración de rutas del proyecto
sys.path.append(os.path.abspath(os.path.join("..")))

from src.config import DatasetKeys, Paths, FeatureConfig
from src.model import ModeloFisico
Paths.init_project()

In [ ]:
# 1. UTILIZACIÓN DE LA LÓGICA DEL PROCESADOR
print("1. Aplicando procesador ModeloFisico (Fourier + ML)...")

# Cargamos CSV ya pre-tratado en fases previas o crudo si lo pasamos por todo el pipeline
df_base = pd.read_csv(Paths.PROC_CSV_AMAEM_NOT_SCALED)
# Ejecutamos la clase instanciada
df_final, _, _ = ModeloFisico.process(df_base, FeatureConfig.PIPELINE_FEATURES)


In [ ]:
# 2. AGRUPACIÓN VISUAL POR BARRIOS (Solo para presentación)
df_final['fecha_mes'] = pd.to_datetime(df_final[DatasetKeys.FECHA]).dt.to_period('M').astype(str)

df_plot = df_final.groupby(['fecha_mes', DatasetKeys.BARRIO]).agg({
    DatasetKeys.CONSUMO_RATIO: 'sum',
    DatasetKeys.PREDICCION_FOURIER: 'sum',
    DatasetKeys.CONSUMO_FISICO_ESPERADO: 'sum'
}).reset_index()
print(f"Total de registros listos: {len(df_plot)} (Meses x Barrios)")

In [ ]:
# 3. MÉTRICAS
print("\n--- MÉTRICAS GLOBALES ---")
print(f"Precisión de Fourier (Solo matemáticas): {r2_score(df_plot[DatasetKeys.CONSUMO_RATIO], df_plot[DatasetKeys.PREDICCION_FOURIER]):.4f}")
print(f"Precisión Híbrida (IA + Exógenas + Barrios): {r2_score(df_plot[DatasetKeys.CONSUMO_RATIO], df_plot[DatasetKeys.CONSUMO_FISICO_ESPERADO]):.4f}")

In [ ]:
# 4. GRÁFICA PARA LA PRESENTACIÓN
barrio_ejemplo = df_plot[DatasetKeys.BARRIO].unique()[1] 
df_graph = df_plot[df_plot[DatasetKeys.BARRIO] == barrio_ejemplo]

plt.figure(figsize=(14, 6))
plt.plot(df_graph['fecha_mes'], df_graph[DatasetKeys.CONSUMO_RATIO], 'ko-', label=f'Consumo Real ({barrio_ejemplo})', linewidth=2)
plt.plot(df_graph['fecha_mes'], df_graph[DatasetKeys.PREDICCION_FOURIER], 'b--', alpha=0.5, label='Física (Curva específica del barrio)')
plt.plot(df_graph['fecha_mes'], df_graph[DatasetKeys.CONSUMO_FISICO_ESPERADO], 'r-', linewidth=3, label='Híbrido (Matemática + Machine Learning)')

plt.title(f'Predicción Hídrica Hiper-segmentada: Impacto Exógeno en {barrio_ejemplo}', fontsize=16)
plt.xticks(rotation=45)
plt.ylabel('Consumo de Agua')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()